### Structured Output

## Pydantic

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile",temperature=0)
model.invoke("How are you?")

AIMessage(content="I'm just a language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 39, 'total_tokens': 85, 'completion_time': 0.104713555, 'completion_tokens_details': None, 'prompt_time': 0.005676077, 'prompt_tokens_details': None, 'queue_time': 0.162847762, 'total_time': 0.110389632}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea185-caee-7371-a66d-e41c73727b4d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 46, 'total_tokens': 85})

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="Director of the movie")
    rating: float = Field(description="Movie ratings out of 10")


In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000024955A08130>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000024955943CD0>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'rating': {'description

In [5]:
model.invoke("Provide the details of the movie Inception")

AIMessage(content='**Inception (2010) Movie Details**\n\n**Plot:**\nInception is a science fiction action film written and directed by Christopher Nolan. The movie follows Cobb (Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (Ken Watanabe) to perform a task known as "inception" - planting an idea in someone\'s mind instead of stealing one.\n\nSaito wants Cobb to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s name, which is wanted by the authorities, and allow him to return to the United States to see his children.\n\nCobb assembles a team of experts, including Arthur (Joseph Gordon-Levitt), Ariadne (Ellen Page), Eames (Tom Hardy), and Saito, to help him perform the inception. The team plans to enter Fischer\'s dreams and convince him to break up his father\'s company

In [6]:
model_with_structure.invoke("Provide the details of the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

### Message output alongside parsed structure

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="This year the movie was released")
    director: str = Field(..., description="Director of the movie")
    rating: float = Field(..., description="Movie ratings out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

resp = model_with_structure.invoke("Provide the details of the movie Inception")
resp

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9rx2zz95r', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 288, 'total_tokens': 320, 'completion_time': 0.052245969, 'completion_tokens_details': None, 'prompt_time': 0.014920012, 'prompt_tokens_details': None, 'queue_time': 0.162114105, 'total_time': 0.067165981}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea185-ef4b-74f0-9f33-57065f50767c-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': '9rx2zz95r', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 288, 'output_tokens': 32,

### Nested Structure

In [20]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str
    oscars: int = Field(
        default=0,
        description="Number of Academy Awards won by the actor in real life"
    )

class MovieDetails(BaseModel):
    title: str
    year: str
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(
        None,
        description="Budget in millions USD"
    )

model_with_structure = model.with_structured_output(MovieDetails)

resp = model_with_structure.invoke(
    "Provide the details of the movie Inception"
)

print(resp)

title='Inception' year='2010' cast=[Actor(name='Leonardo DiCaprio', role='Cobb', oscars=1), Actor(name='Joseph Gordon-Levitt', role='Arthur', oscars=0)] genres=['Action', 'Sci-Fi', 'Thriller'] budget=160.0


## TypedDict

In [23]:
from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "This year the movie was released"]
    director: Annotated[str, ..., "Director of the movie"]
    rating: Annotated[float, ..., "Movie ratings out of 10"]

m1 = model.with_structured_output(Movie)

m1.invoke("Provide the details of the movie Avenger")

{'director': 'Joss Whedon', 'rating': 8.1, 'title': 'Avenger', 'year': 2012}

In [24]:
class Actor(TypedDict):
    name: str
    role: str
    oscars: Annotated[int, ..., "Number of Academy Awards won by the actor in real life"]

class MovieDetails(TypedDict):
    title: str
    year: str
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(
        None,
        description="Budget in millions USD"
    )

model_with_structure = model.with_structured_output(MovieDetails)

resp = model_with_structure.invoke(
    "Provide the details of the movie Inception"
)

print(resp)

{'budget': 160000000, 'cast': [{'name': 'Leonardo DiCaprio', 'oscars': 1, 'role': 'Cobb'}, {'name': 'Joseph Gordon-Levitt', 'oscars': 0, 'role': 'Arthur'}, {'name': 'Ellen Page', 'oscars': 0, 'role': 'Ariadne'}], 'genres': ['Action', 'Sci-Fi', 'Thriller'], 'title': 'Inception', 'year': '2010'}


In [25]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

## Data Classes

In [30]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile"
)

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    # always pass the object of the model here 
    model = model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (+91) 8840172561"
    }]
})


result["structured_response"]



ContactInfo(name='John Doe', email='john@example.com', phone='(+91) 8840172561')

In [32]:
### Data Class
# Python automatically generates:
# __init__
# __repr__
# __eq__

from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile"
)

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str 
    email: str
    phone: str 

agent = create_agent(
    # always pass the object of the model here 
    model = model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (+91) 8840172561"
    }]
})


result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(+91) 8840172561')